# Feature Extraction Pipeline

## Project Information
- **Source**: Transformers Packt Course (lazyprogrammer.me/course_files/nlp/)
- **Objective**: Extract embeddings from text for semantic search and similarity
- **Pipeline**: `feature-extraction`
- **Dataset**: BBC News Articles (bbc_text_cls.csv)

## Overview
This notebook demonstrates how to use the feature-extraction pipeline to create embeddings from text, which can be used for semantic search, document similarity, and clustering.


In [ ]:
%pip install transformers pandas numpy matplotlib seaborn scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')


## Initialize Feature Extraction Pipeline


In [ ]:
# Initialize the feature extraction pipeline
featurizer = pipeline('feature-extraction')
print(f"Pipeline initialized: {type(featurizer)}")

# Test with a simple example
test_text = "this is my text"
results = featurizer(test_text)
print(f"\nResult type: {type(results)}")
print(f"Result length: {len(results)}")
print(f"First token embedding shape: {np.array(results[0]).shape}")


## Load and Prepare Dataset


In [ ]:
# Load BBC news articles dataset
df = pd.read_csv('bbc_text_cls.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nUnique labels: {df['labels'].unique()}")
df.head()


## Extract Embeddings from Documents


In [ ]:
# Process texts: truncate to first 100 words for efficiency
texts = [' '.join(text.split()[:100]) for text in df['text']]
print(f"Processing {len(texts)} documents...")
print(f"Sample text (first 200 chars): {texts[0][:200]}...")


In [ ]:
# Extract embeddings for all documents
# Note: This may take a while for large datasets
embeddings = featurizer(texts)

# Extract the first token's embedding (CLS token) for each document
# This represents the entire document
embeddings = [e[0][0] for e in embeddings]  # [batch][token][embedding_dim]
embeddings = np.array(embeddings)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Each document is represented by a {embeddings.shape[1]}-dimensional vector")


## Build Semantic Search with Nearest Neighbors


In [ ]:
# Build a nearest neighbors model for semantic search
# Using cosine similarity (dot product of normalized vectors)
model = NearestNeighbors(metric='cosine', n_neighbors=5)
model.fit(embeddings)
print("Nearest Neighbors model fitted successfully")


## Example: Find Similar Documents


In [ ]:
# Find documents similar to a specific document (e.g., document at index 10)
query_idx = 10
query_embedding = embeddings[query_idx].reshape(1, -1)

distances, indices = model.kneighbors(query_embedding)
print(f"Query document (index {query_idx}):")
print(f"Label: {df.iloc[query_idx]['labels']}")
print(f"Text preview: {df.iloc[query_idx]['text'][:200]}...")
print(f"\nMost similar documents:")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    if idx != query_idx:  # Skip the query document itself
        print(f"\n{i+1}. Document {idx} (distance: {dist:.4f})")
        print(f"   Label: {df.iloc[idx]['labels']}")
        print(f"   Text preview: {df.iloc[idx]['text'][:200]}...")


## Example: Search with Custom Query


In [ ]:
# Search for documents similar to a custom query
query_text = "TV and film"
print(f"Query: '{query_text}'")

# Extract embedding for the query
query_embedding = featurizer(query_text)
query_embedding = np.array(query_embedding[0][0]).reshape(1, -1)

# Find nearest neighbors
distances, indices = model.kneighbors(query_embedding)

print(f"\nTop 5 most similar documents:")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"\n{i+1}. Document {idx} (distance: {dist:.4f})")
    print(f"   Label: {df.iloc[idx]['labels']}")
    print(f"   Text preview: {df.iloc[idx]['text'][:200]}...")


## Results Summary

The feature extraction pipeline creates dense vector representations (embeddings) of text that capture semantic meaning. These embeddings can be used for:

**Applications:**
- **Semantic Search**: Find documents similar to a query
- **Document Clustering**: Group similar documents together
- **Recommendation Systems**: Recommend similar content
- **Information Retrieval**: Improve search beyond keyword matching

**Key Features:**
- Each document is represented as a fixed-size vector
- Embeddings capture semantic relationships
- Cosine similarity works well for comparing embeddings
